# 03 - Modelado de clasificación

## Objetivo

En esta fase se desarrollarán modelos de Machine Learning capaces de clasificar automáticamente una consulta textual en función de la categoría de producto a la que pertenece.

El problema se plantea como una tarea de **clasificación supervisada multiclase**, donde:

- **Variable predictora (X):** `Consumer complaint narrative`
- **Variable objetivo (y):** `Product`

El dataset utilizado procede de la fase previa de análisis y preparación, en la que se seleccionó un periodo con una taxonomía homogénea, se eliminaron narrativas con etiquetas contradictorias y se conservaron únicamente textos únicos.

El objetivo será comenzar con un modelo base y posteriormente comparar diferentes estrategias de representación y clasificación.

## 1. Importación de librerías

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 2. Carga del dataset procesado

Se carga el conjunto de datos generado durante la etapa de preparación. Esta versión contiene únicamente las observaciones seleccionadas para el proceso de modelado y permite trabajar directamente sobre un corpus previamente depurado.

In [2]:
ruta_datos = "../data/processed/complaints_model.csv"

df = pd.read_csv(ruta_datos)

print("Dimensiones:", df.shape)

print("\nColumnas:")
print(df.columns.tolist())

df.head()

Dimensiones: (1299353, 3)

Columnas:
['Date received', 'Product', 'Consumer complaint narrative']


,Date received,Product,Consumer complaint narrative
0,2023-11-15,Credit reporting or other personal consumer re...,I have a long time victim of fraud and identit...
1,2024-02-12,Checking or savings account,From XXXX XXXX through XXXX XXXX my checking a...
2,2025-11-24,"Money transfer, virtual currency, or money ser...","What happened? In XX/XX/year>, XXXX XXXX faile..."
3,2025-12-03,Mortgage,"On XXXX, received a Loan Estimate from XXXX XX..."
4,2025-12-03,Mortgage,Subject : Request for Investigation Into Mortg...


## 3. Definición de las variables predictora y objetivo

Para el problema de clasificación se utiliza el contenido textual de cada reclamación como variable predictora (`X`) y la categoría de producto asociada como variable objetivo (`y`).

La fecha de recepción se conserva en el dataset procesado como información auxiliar, pero no se incorpora como variable predictora. De esta forma, el modelo deberá aprender a clasificar las consultas exclusivamente a partir de su contenido textual.

In [3]:
X = df["Consumer complaint narrative"]
y = df["Product"]

print("Número de observaciones en X:", len(X))
print("Número de observaciones en y:", len(y))
print("Número de clases:", y.nunique())

print("\nEjemplo de X:")
print(X.iloc[0])

print("\nEtiqueta y asociada:")
print(y.iloc[0])

Número de observaciones en X: 1299353
Número de observaciones en y: 1299353
Número de clases: 11

Ejemplo de X:
I have a long time victim of fraud and identity theft with police reports confirmation and confirmation from some organizations that I have been a victim of with people who stole my information opened credit cards accounts in my name and phone accounts which I reported to the credit bureaus which open fraudulent accounts without my consent by ignoring the fraud alert on my credit file and banks having credit cards under my name which I reported to the credit bureaus to remove but they refused and asking me to bring the police report which these organizations refuse to respect due to my race and XXXX with lack of protection I contacted the banks to give me the credit cards appearing on my credit file they refused and asked me to contact the credit bureaus so i I have been given a run around with various complaints filed and police reports which have been ignored I contacted th

## 4. División en conjuntos de entrenamiento y prueba

Para evaluar la capacidad de generalización del modelo, el dataset se divide en dos subconjuntos independientes:

- **Conjunto de entrenamiento (80 %):** utilizado para aprender la relación entre las narrativas y las categorías de producto.
- **Conjunto de prueba (20 %):** reservado para evaluar el modelo sobre observaciones que no han sido utilizadas durante el entrenamiento.

La división se realiza de forma estratificada, manteniendo aproximadamente la misma distribución de las categorías de `Product` en ambos conjuntos.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Observaciones totales:", len(X))
print("Entrenamiento:", len(X_train))
print("Prueba:", len(X_test))

print(
    "\nPorcentaje entrenamiento:",
    round(len(X_train) / len(X) * 100, 2),
    "%"
)

print(
    "Porcentaje prueba:",
    round(len(X_test) / len(X) * 100, 2),
    "%"
)

Observaciones totales: 1299353
Entrenamiento: 1039482
Prueba: 259871

Porcentaje entrenamiento: 80.0 %
Porcentaje prueba: 20.0 %


## 5. Comprobación de la distribución de clases

Tras realizar la división, se comprueba que la distribución de las categorías de producto se mantiene aproximadamente constante entre los conjuntos de entrenamiento y prueba.

Esta comprobación es especialmente relevante debido al desbalance existente entre las clases.

In [5]:
distribucion_train = (
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

distribucion_test = (
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

comparacion_distribucion = pd.DataFrame({
    "Train (%)": distribucion_train,
    "Test (%)": distribucion_test
})

comparacion_distribucion

,Train (%),Test (%)
Product,,
Credit reporting or other personal consumer reports,57.81,57.81
Debt collection,12.51,12.51
Checking or savings account,7.99,7.99
Credit card,7.49,7.49
"Money transfer, virtual currency, or money service",4.77,4.77
Mortgage,2.84,2.84
Vehicle loan or lease,2.03,2.03
Student loan,2.00,2.00
"Payday loan, title loan, personal loan, or advance loan",1.38,1.38


## 6. Modelo baseline

Antes de desarrollar modelos de clasificación basados en el contenido textual, se establece un modelo de referencia (*baseline*).

Este modelo asigna a todas las observaciones la categoría más frecuente del conjunto de entrenamiento, sin analizar el contenido de las narrativas.

Su rendimiento representa un punto de referencia mínimo que los modelos posteriores deberán superar. Debido al desbalance existente entre las clases, además de la exactitud (*accuracy*), se utiliza el F1-score macro, que otorga el mismo peso a cada categoría independientemente de su frecuencia.

In [6]:
clase_mayoritaria = y_train.value_counts().idxmax()

print("Clase mayoritaria:")
print(clase_mayoritaria)

y_pred_baseline = np.full(
    len(y_test),
    clase_mayoritaria
)

accuracy_baseline = accuracy_score(
    y_test,
    y_pred_baseline
)

f1_macro_baseline = f1_score(
    y_test,
    y_pred_baseline,
    average="macro"
)

print(
    "\nAccuracy baseline:",
    round(accuracy_baseline, 4)
)

print(
    "F1-score macro baseline:",
    round(f1_macro_baseline, 4)
)

Clase mayoritaria:
Credit reporting or other personal consumer reports

Accuracy baseline: 0.5781
F1-score macro baseline: 0.0666


### Resultado del baseline

El modelo baseline obtiene una **accuracy del 57,81 %**, resultado que coincide con la proporción de la clase mayoritaria en el conjunto de prueba. Este comportamiento era esperado, ya que el modelo asigna todas las observaciones a dicha categoría.

Sin embargo, el **F1-score macro es únicamente 0,0666**, lo que evidencia que el modelo no presenta capacidad real para distinguir entre las diferentes categorías.

Estos resultados refuerzan la necesidad de utilizar métricas adicionales a la accuracy en problemas con clases desbalanceadas y establecen el rendimiento mínimo de referencia para los modelos posteriores.

## 7. Representación del texto mediante TF-IDF

Los algoritmos tradicionales de Machine Learning no pueden procesar directamente texto en lenguaje natural. Por este motivo, es necesario transformar las narrativas en una representación numérica.

En esta primera aproximación se utiliza **TF-IDF (Term Frequency-Inverse Document Frequency)**, una técnica que asigna un peso a cada término teniendo en cuenta tanto su frecuencia dentro de un documento como su frecuencia en el conjunto del corpus.

De esta forma, las palabras especialmente frecuentes en todos los documentos reciben menor peso, mientras que aquellas que resultan más características de determinados textos adquieren una mayor relevancia.

### 7.1 Creación de una muestra de entrenamiento

Debido al elevado tamaño del corpus, se utiliza inicialmente una muestra estratificada del conjunto de entrenamiento para desarrollar y comparar los primeros modelos de forma eficiente.

Esta estrategia permite reducir el coste computacional durante la fase experimental, manteniendo la distribución original de las categorías. Una vez seleccionada la configuración más adecuada, podrá evaluarse su aplicación sobre un volumen mayor de datos.

In [7]:
X_train_exp, _, y_train_exp, _ = train_test_split(
    X_train,
    y_train,
    train_size=100000,
    random_state=42,
    stratify=y_train
)

print("Tamaño del entrenamiento completo:", len(X_train))
print("Tamaño de la muestra experimental:", len(X_train_exp))

print("\nDistribución de la muestra:")
print(
    y_train_exp.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Tamaño del entrenamiento completo: 1039482
Tamaño de la muestra experimental: 100000

Distribución de la muestra:
Product
Credit reporting or other personal consumer reports        57.80
Debt collection                                            12.51
Checking or savings account                                 7.99
Credit card                                                 7.49
Money transfer, virtual currency, or money service          4.77
Mortgage                                                    2.84
Vehicle loan or lease                                       2.03
Student loan                                                2.00
Payday loan, title loan, personal loan, or advance loan     1.38
Prepaid card                                                0.77
Debt or credit management                                   0.41
Name: proportion, dtype: float64


### 7.2 Vectorización mediante TF-IDF

Para transformar las narrativas en variables numéricas se utiliza `TfidfVectorizer`.

En esta primera configuración se consideran unigramas y bigramas, permitiendo representar tanto palabras individuales como combinaciones de dos términos consecutivos. Además, se eliminan términos extremadamente infrecuentes y se limita el tamaño máximo del vocabulario para controlar el coste computacional.

El vectorizador se ajusta exclusivamente sobre el conjunto de entrenamiento experimental, evitando utilizar información procedente del conjunto de prueba durante el aprendizaje de la representación.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_features=50000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train_exp)

print("Dimensiones de la matriz TF-IDF:", X_train_tfidf.shape)
print("Número de términos del vocabulario:", len(tfidf.vocabulary_))

Dimensiones de la matriz TF-IDF: (100000, 50000)
Número de términos del vocabulario: 50000


### 7.3 Inspección del vocabulario generado

Una vez ajustado el vectorizador, se inspeccionan algunos de los términos que forman parte del vocabulario generado.

La matriz resultante contiene una fila por cada narrativa y una columna por cada término o combinación de términos seleccionada. Los valores almacenados representan el peso TF-IDF de cada término dentro de cada documento.

In [10]:
terminos = tfidf.get_feature_names_out()

print("Número total de términos:", len(terminos))

print("\nPrimeros 30 términos:")
print(terminos[:30])

Número total de términos: 50000

Primeros 30 términos:
['00' '00 00' '00 10' '00 100' '00 1000' '00 10000' '00 1100' '00 120'
 '00 1200' '00 1300' '00 1400' '00 15' '00 150' '00 1500' '00 1800'
 '00 200' '00 2000' '00 250' '00 2500' '00 30' '00 300' '00 3000' '00 400'
 '00 50' '00 500' '00 5000' '00 60' '00 90' '00 able' '00 accordance']


In [11]:
indice = 0

texto_ejemplo = X_train_exp.iloc[indice]
vector_ejemplo = X_train_tfidf[indice]

indices_no_cero = vector_ejemplo.nonzero()[1]

pesos_ejemplo = pd.DataFrame({
    "termino": terminos[indices_no_cero],
    "peso_tfidf": vector_ejemplo[0, indices_no_cero].toarray().flatten()
}).sort_values(
    "peso_tfidf",
    ascending=False
)

print("Texto original:\n")
print(texto_ejemplo)

print("\nTérminos con mayor peso TF-IDF:")
pesos_ejemplo.head(15)

Texto original:

I am a victim of identity theft. Please delete or remove these items on my behalf. 
These items are not mine and this is greatly affecting me and my personal life. I request that you block the following information and please let me know if you need any other information from me to block this information from my credit report. Thank you. 




XXXX XXXX Account Number : XXXX XXXX XXXX  XXXX Account Number : XXXX XXXX XXXX  XXXX Account Number : XXXX XXXX XXXX Account Number : XXXX XXXX XXXX Account Number : XXXX XXXX XXXX  Account Number : XXXX XXXX Account Number : XXXX Inquiry XXXX XXXX XXXX XXXX

Términos con mayor peso TF-IDF:


,termino,peso_tfidf
52,account number,0.225517
53,number xxxx,0.214786
51,xxxx account,0.195128
31,behalf items,0.192747
32,items greatly,0.192747
30,items behalf,0.192345
36,life request,0.191181
33,greatly affecting,0.189363
34,affecting personal,0.185222
40,information let,0.181228


## 8. Primer modelo: Regresión Logística

Una vez obtenida la representación TF-IDF de las narrativas, se entrena un primer modelo supervisado de clasificación.

Se utiliza **Regresión Logística**, un algoritmo ampliamente empleado como modelo base en problemas de clasificación de texto debido a su eficiencia con representaciones de alta dimensionalidad y dispersas como TF-IDF.

El modelo aprenderá, a partir de las narrativas del conjunto de entrenamiento experimental, qué términos y combinaciones de términos están asociados con cada una de las categorías de `Product`.

In [12]:
from sklearn.linear_model import LogisticRegression

### 8.1 Entrenamiento

In [13]:
modelo_lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

modelo_lr.fit(
    X_train_tfidf,
    y_train_exp
)

print("Modelo entrenado correctamente.")

Modelo entrenado correctamente.


### 8.2 Evaluación sobre el conjunto de prueba

Una vez entrenado el modelo, se evalúa su capacidad de generalización utilizando el conjunto de prueba, formado por observaciones que no participaron en el entrenamiento.

Las narrativas del conjunto de prueba se transforman utilizando el vectorizador TF-IDF previamente ajustado sobre los datos de entrenamiento. De esta forma, no se incorpora información del conjunto de prueba durante el aprendizaje y se evita la fuga de información (*data leakage*).

Posteriormente, el modelo genera una categoría de producto para cada narrativa y se comparan estas predicciones con las etiquetas reales.

In [14]:
X_test_tfidf = tfidf.transform(X_test)

print(
    "Dimensiones del conjunto de prueba vectorizado:",
    X_test_tfidf.shape
)

Dimensiones del conjunto de prueba vectorizado: (259871, 50000)


In [15]:
y_pred_lr = modelo_lr.predict(X_test_tfidf)

print("Predicciones generadas:", len(y_pred_lr))

Predicciones generadas: 259871


In [16]:
accuracy_lr = accuracy_score(
    y_test,
    y_pred_lr
)

f1_macro_lr = f1_score(
    y_test,
    y_pred_lr,
    average="macro"
)

print(
    "Accuracy Regresión Logística:",
    round(accuracy_lr, 4)
)

print(
    "F1-score macro Regresión Logística:",
    round(f1_macro_lr, 4)
)

print("\n--- Comparación con baseline ---")

print(
    "Baseline accuracy:",
    round(accuracy_baseline, 4)
)

print(
    "Regresión Logística accuracy:",
    round(accuracy_lr, 4)
)

print()

print(
    "Baseline F1 macro:",
    round(f1_macro_baseline, 4)
)

print(
    "Regresión Logística F1 macro:",
    round(f1_macro_lr, 4)
)

Accuracy Regresión Logística: 0.8588
F1-score macro Regresión Logística: 0.6966

--- Comparación con baseline ---
Baseline accuracy: 0.5781
Regresión Logística accuracy: 0.8588

Baseline F1 macro: 0.0666
Regresión Logística F1 macro: 0.6966


### Resultados del primer modelo

La Regresión Logística entrenada sobre la representación TF-IDF obtiene una **accuracy de 0,8588** y un **F1-score macro de 0,6966** sobre el conjunto de prueba.

Estos resultados representan una mejora considerable respecto al modelo baseline, que obtuvo una accuracy de 0,5781 y un F1-score macro de 0,0666.

La mejora del F1-score macro resulta especialmente relevante debido al desbalance existente entre las categorías. El resultado indica que el modelo está aprendiendo patrones textuales asociados a diferentes productos y no se limita a favorecer la clase mayoritaria.

Además, este primer experimento se ha realizado utilizando una muestra estratificada de 100.000 observaciones del conjunto de entrenamiento, por lo que todavía existe margen para analizar el efecto de utilizar un mayor volumen de datos y de optimizar la configuración del modelo.

### 8.3 Análisis del rendimiento por categoría

Las métricas globales permiten evaluar el comportamiento general del modelo, pero no muestran las diferencias existentes entre las distintas categorías.

Por este motivo, se analiza el rendimiento individual de cada clase mediante *precision*, *recall* y *F1-score*. Este análisis resulta especialmente importante en un problema con clases desbalanceadas, ya que permite identificar las categorías que presentan mayores dificultades de clasificación.

In [17]:
reporte_lr = classification_report(
    y_test,
    y_pred_lr,
    output_dict=True,
    zero_division=0
)

reporte_lr_df = pd.DataFrame(reporte_lr).T

reporte_lr_df

,precision,recall,f1-score,support
Checking or savings account,0.773697,0.833221,0.802357,20758.000000
Credit card,0.773126,0.760462,0.766742,19475.000000
Credit reporting or other personal consumer reports,0.901015,0.951158,0.925408,150219.000000
Debt collection,0.778447,0.700145,0.737222,32509.000000
Debt or credit management,0.781250,0.070822,0.129870,1059.000000
"Money transfer, virtual currency, or money service",0.818692,0.753551,0.784772,12392.000000
Mortgage,0.899743,0.853196,0.875852,7384.000000
"Payday loan, title loan, personal loan, or advance loan",0.689929,0.435097,0.533652,3590.000000
Prepaid card,0.879495,0.451822,0.596966,2003.000000
Student loan,0.883778,0.757937,0.816035,5197.000000


### Interpretación del rendimiento por clase

El análisis por categoría muestra que el rendimiento del modelo no es homogéneo. Las clases con mayor representación presentan, en general, resultados elevados. Destaca especialmente `Credit reporting or other personal consumer reports`, con un F1-score de aproximadamente 0,93, así como `Mortgage`, con un F1-score próximo a 0,88.

Sin embargo, el rendimiento disminuye en algunas categorías minoritarias. El caso más significativo es `Debt or credit management`, que presenta una precisión de 0,78 pero un recall de únicamente 0,07. Esto indica que, aunque las predicciones realizadas para esta categoría suelen ser correctas, el modelo solo consigue identificar una pequeña proporción de los casos que realmente pertenecen a ella.

Este comportamiento evidencia el impacto del desbalance de clases y explica la diferencia observada entre el F1-score weighted (0,85) y el F1-score macro (0,70). Por tanto, la accuracy global no resulta suficiente para evaluar el sistema y será necesario considerar estrategias destinadas a mejorar el reconocimiento de las categorías minoritarias.

## 9. Tratamiento del desbalance de clases

El análisis del primer modelo muestra diferencias importantes de rendimiento entre las categorías. En particular, algunas clases minoritarias presentan valores reducidos de recall, lo que indica que una parte significativa de sus observaciones no está siendo correctamente identificada.

Para analizar el efecto del desbalance se entrena una segunda Regresión Logística utilizando ponderación automática de clases mediante `class_weight="balanced"`.

Esta configuración asigna un mayor peso a las clases menos frecuentes y un menor peso a las más representadas. El objetivo es comprobar si se mejora la capacidad del modelo para identificar categorías minoritarias, especialmente en términos de recall y F1-score macro.

### 9.1 Regresión Logística con clases balanceadas

In [18]:
modelo_lr_balanced = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight="balanced"
)

modelo_lr_balanced.fit(
    X_train_tfidf,
    y_train_exp
)

print("Modelo balanceado entrenado correctamente.")

Modelo balanceado entrenado correctamente.


### 9.2 Predicción y comparación

In [19]:
y_pred_lr_balanced = modelo_lr_balanced.predict(
    X_test_tfidf
)

accuracy_lr_balanced = accuracy_score(
    y_test,
    y_pred_lr_balanced
)

f1_macro_lr_balanced = f1_score(
    y_test,
    y_pred_lr_balanced,
    average="macro"
)

print(
    "Accuracy modelo balanceado:",
    round(accuracy_lr_balanced, 4)
)

print(
    "F1 macro modelo balanceado:",
    round(f1_macro_lr_balanced, 4)
)

Accuracy modelo balanceado: 0.8268
F1 macro modelo balanceado: 0.6943


In [20]:
comparacion_modelos = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Regresión Logística",
        "Regresión Logística balanceada"
    ],
    "Accuracy": [
        accuracy_baseline,
        accuracy_lr,
        accuracy_lr_balanced
    ],
    "F1 macro": [
        f1_macro_baseline,
        f1_macro_lr,
        f1_macro_lr_balanced
    ]
})

comparacion_modelos[["Accuracy", "F1 macro"]] = (
    comparacion_modelos[["Accuracy", "F1 macro"]]
    .round(4)
)

comparacion_modelos

,Modelo,Accuracy,F1 macro
0,Baseline,0.5781,0.0666
1,Regresión Logística,0.8588,0.6966
2,Regresión Logística balanceada,0.8268,0.6943


### 9.3 Evaluación del efecto de la ponderación de clases

Los resultados globales muestran que la incorporación de pesos balanceados no mejora el rendimiento general del modelo. La accuracy disminuye de 0,8588 a 0,8268 y el F1-score macro pasa de 0,6966 a 0,6943.

No obstante, dado que el objetivo de esta estrategia es favorecer el reconocimiento de las clases minoritarias, resulta necesario analizar el rendimiento individual por categoría antes de determinar si la ponderación de clases aporta algún beneficio. 

In [21]:
reporte_balanced = classification_report(
    y_test,
    y_pred_lr_balanced,
    output_dict=True,
    zero_division=0
)

reporte_balanced_df = pd.DataFrame(reporte_balanced).T

reporte_balanced_df

,precision,recall,f1-score,support
Checking or savings account,0.789295,0.797090,0.793174,20758.000000
Credit card,0.696197,0.798973,0.744053,19475.000000
Credit reporting or other personal consumer reports,0.959041,0.849340,0.900863,150219.000000
Debt collection,0.667435,0.783475,0.720815,32509.000000
Debt or credit management,0.208255,0.314448,0.250564,1059.000000
"Money transfer, virtual currency, or money service",0.774002,0.797611,0.785629,12392.000000
Mortgage,0.811455,0.913326,0.859382,7384.000000
"Payday loan, title loan, personal loan, or advance loan",0.435770,0.684123,0.532408,3590.000000
Prepaid card,0.562674,0.705941,0.626218,2003.000000
Student loan,0.684133,0.899365,0.777122,5197.000000


In [22]:
clases = y_test.unique()

comparacion_clases = pd.DataFrame({
    "F1 original": [
        reporte_lr[clase]["f1-score"]
        for clase in clases
    ],
    "F1 balanceado": [
        reporte_balanced[clase]["f1-score"]
        for clase in clases
    ],
    "Recall original": [
        reporte_lr[clase]["recall"]
        for clase in clases
    ],
    "Recall balanceado": [
        reporte_balanced[clase]["recall"]
        for clase in clases
    ]
}, index=clases)

comparacion_clases["Diferencia F1"] = (
    comparacion_clases["F1 balanceado"]
    - comparacion_clases["F1 original"]
)

comparacion_clases["Diferencia Recall"] = (
    comparacion_clases["Recall balanceado"]
    - comparacion_clases["Recall original"]
)

comparacion_clases.round(3).sort_values(
    "Diferencia F1",
    ascending=False
)

,F1 original,F1 balanceado,Recall original,Recall balanceado,Diferencia F1,Diferencia Recall
Debt or credit management,0.130,0.251,0.071,0.314,0.121,0.244
Prepaid card,0.597,0.626,0.452,0.706,0.029,0.254
"Money transfer, virtual currency, or money service",0.785,0.786,0.754,0.798,0.001,0.044
"Payday loan, title loan, personal loan, or advance loan",0.534,0.532,0.435,0.684,-0.001,0.249
Checking or savings account,0.802,0.793,0.833,0.797,-0.009,-0.036
Mortgage,0.876,0.859,0.853,0.913,-0.016,0.060
Debt collection,0.737,0.721,0.700,0.783,-0.016,0.083
Credit card,0.767,0.744,0.760,0.799,-0.023,0.039
Credit reporting or other personal consumer reports,0.925,0.901,0.951,0.849,-0.025,-0.102
Student loan,0.816,0.777,0.758,0.899,-0.039,0.141


### Conclusión del experimento de balanceo

La ponderación de clases produce un aumento considerable del recall en varias de las categorías minoritarias. Por ejemplo, `Debt or credit management` pasa de un recall de 0,071 a 0,314, mientras que `Prepaid card` aumenta de 0,452 a 0,706 y la categoría de `Payday loan` pasa de 0,435 a 0,684.

Sin embargo, este incremento de sensibilidad hacia las clases menos frecuentes se produce a costa de una reducción de la precisión y del rendimiento en otras categorías. Como consecuencia, la accuracy global disminuye de 0,8588 a 0,8268 y el F1-score macro pasa de 0,6966 a 0,6943.

Por tanto, aunque la ponderación mediante `class_weight="balanced"` mejora la capacidad de detección de determinadas clases minoritarias, no produce una mejora global del F1-score macro. En esta fase se mantiene la Regresión Logística sin ponderación como modelo de referencia con mejor rendimiento global.

## 10. Segundo modelo: Linear SVM

Con el objetivo de comparar el rendimiento de diferentes algoritmos de clasificación, se entrena un segundo modelo utilizando una Máquina de Vectores de Soporte lineal (`LinearSVC`).

Los modelos SVM lineales son especialmente adecuados para problemas de clasificación de texto representados mediante TF-IDF, donde el número de características es elevado y las matrices presentan una estructura dispersa.

Para garantizar una comparación consistente con la Regresión Logística, se mantienen la misma muestra de entrenamiento, la misma representación TF-IDF y el mismo conjunto de prueba. De esta forma, la principal diferencia entre ambos experimentos corresponde al algoritmo de clasificación utilizado.

### 10.1 Entrenamiento

In [23]:
from sklearn.svm import LinearSVC

In [24]:
modelo_svm = LinearSVC(
    random_state=42
)

modelo_svm.fit(
    X_train_tfidf,
    y_train_exp
)

print("Modelo Linear SVM entrenado correctamente.")

Modelo Linear SVM entrenado correctamente.


### 10.2 Evaluación

In [25]:
y_pred_svm = modelo_svm.predict(
    X_test_tfidf
)

accuracy_svm = accuracy_score(
    y_test,
    y_pred_svm
)

f1_macro_svm = f1_score(
    y_test,
    y_pred_svm,
    average="macro"
)

print(
    "Accuracy Linear SVM:",
    round(accuracy_svm, 4)
)

print(
    "F1 macro Linear SVM:",
    round(f1_macro_svm, 4)
)

Accuracy Linear SVM: 0.8554
F1 macro Linear SVM: 0.7118


In [26]:
comparacion_modelos = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Regresión Logística",
        "Regresión Logística balanceada",
        "Linear SVM"
    ],
    "Accuracy": [
        accuracy_baseline,
        accuracy_lr,
        accuracy_lr_balanced,
        accuracy_svm
    ],
    "F1 macro": [
        f1_macro_baseline,
        f1_macro_lr,
        f1_macro_lr_balanced,
        f1_macro_svm
    ]
})

comparacion_modelos[["Accuracy", "F1 macro"]] = (
    comparacion_modelos[["Accuracy", "F1 macro"]]
    .round(4)
)

comparacion_modelos.sort_values(
    "F1 macro",
    ascending=False
)

,Modelo,Accuracy,F1 macro
3,Linear SVM,0.8554,0.7118
1,Regresión Logística,0.8588,0.6966
2,Regresión Logística balanceada,0.8268,0.6943
0,Baseline,0.5781,0.0666


### Comparación entre Regresión Logística y Linear SVM

El modelo Linear SVM obtiene una accuracy de **0,8554** y un F1-score macro de **0,7118**.

En comparación con la Regresión Logística, la accuracy disminuye ligeramente de 0,8588 a 0,8554. Sin embargo, el F1-score macro aumenta de 0,6966 a 0,7118.

Dado el importante desbalance existente entre las categorías, el F1-score macro constituye una métrica especialmente relevante, ya que asigna el mismo peso al rendimiento de cada clase independientemente de su frecuencia.

Por este motivo, Linear SVM se considera, hasta este punto del análisis, el modelo con mejor equilibrio global entre las diferentes categorías, aunque la Regresión Logística presente una accuracy ligeramente superior.

### 10.3 Rendimiento del SVM por categoría

In [27]:
reporte_svm = classification_report(
    y_test,
    y_pred_svm,
    output_dict=True,
    zero_division=0
)

reporte_svm_df = pd.DataFrame(reporte_svm).T

reporte_svm_df

,precision,recall,f1-score,support
Checking or savings account,0.776258,0.803931,0.789852,20758.000000
Credit card,0.764369,0.747728,0.755957,19475.000000
Credit reporting or other personal consumer reports,0.908676,0.943343,0.925685,150219.000000
Debt collection,0.757877,0.704420,0.730171,32509.000000
Debt or credit management,0.677165,0.162417,0.261995,1059.000000
"Money transfer, virtual currency, or money service",0.785233,0.761217,0.773038,12392.000000
Mortgage,0.876750,0.873781,0.875263,7384.000000
"Payday loan, title loan, personal loan, or advance loan",0.646526,0.476880,0.548894,3590.000000
Prepaid card,0.804522,0.550674,0.653823,2003.000000
Student loan,0.855807,0.792573,0.822977,5197.000000


In [28]:
comparacion_lr_svm = pd.DataFrame({
    "F1 Regresión Logística": [
        reporte_lr[clase]["f1-score"]
        for clase in clases
    ],
    "F1 Linear SVM": [
        reporte_svm[clase]["f1-score"]
        for clase in clases
    ],
    "Recall Regresión Logística": [
        reporte_lr[clase]["recall"]
        for clase in clases
    ],
    "Recall Linear SVM": [
        reporte_svm[clase]["recall"]
        for clase in clases
    ]
}, index=clases)

comparacion_lr_svm["Diferencia F1"] = (
    comparacion_lr_svm["F1 Linear SVM"]
    - comparacion_lr_svm["F1 Regresión Logística"]
)

comparacion_lr_svm.round(3).sort_values(
    "Diferencia F1",
    ascending=False
)

,F1 Regresión Logística,F1 Linear SVM,Recall Regresión Logística,Recall Linear SVM,Diferencia F1
Debt or credit management,0.130,0.262,0.071,0.162,0.132
Prepaid card,0.597,0.654,0.452,0.551,0.057
"Payday loan, title loan, personal loan, or advance loan",0.534,0.549,0.435,0.477,0.015
Student loan,0.816,0.823,0.758,0.793,0.007
Credit reporting or other personal consumer reports,0.925,0.926,0.951,0.943,0.000
Mortgage,0.876,0.875,0.853,0.874,-0.001
Vehicle loan or lease,0.694,0.692,0.629,0.653,-0.002
Debt collection,0.737,0.730,0.700,0.704,-0.007
Credit card,0.767,0.756,0.760,0.748,-0.011
"Money transfer, virtual currency, or money service",0.785,0.773,0.754,0.761,-0.012


### Interpretación de la comparación por categoría

La comparación entre ambos modelos permite observar que la mejora del F1-score macro obtenida por Linear SVM procede principalmente de un mejor comportamiento en algunas de las categorías con mayor dificultad de clasificación.

El caso más significativo corresponde a `Debt or credit management`, cuyo F1-score aumenta de 0,130 con Regresión Logística a 0,262 con Linear SVM. Esta mejora está acompañada por un incremento del recall de 0,071 a 0,162. También se observan mejoras en `Prepaid card`, cuyo F1-score pasa de 0,597 a 0,654, y en la categoría de `Payday loan`, que aumenta de 0,534 a 0,549.

Por el contrario, Linear SVM presenta pequeñas reducciones de rendimiento en algunas categorías, como `Checking or savings account`, `Money transfer, virtual currency, or money service` y `Credit card`. Estas pérdidas son relativamente reducidas frente a las mejoras obtenidas en algunas de las clases más problemáticas.

En conjunto, estos resultados explican el aumento del F1-score macro de 0,6966 a 0,7118. Por tanto, se selecciona **Linear SVM como el mejor modelo de esta primera fase experimental**, al presentar un mejor equilibrio de clasificación entre las diferentes categorías.

## 11. Tercer modelo: Complement Naive Bayes

Como tercer algoritmo de clasificación se utiliza **Complement Naive Bayes**, una variante de Naive Bayes especialmente diseñada para problemas de clasificación de texto y conjuntos de datos con clases desbalanceadas.

El modelo trabaja adecuadamente con representaciones TF-IDF y presenta un coste computacional reducido. Para mantener la comparabilidad con los experimentos anteriores, se utilizan la misma muestra de entrenamiento, la misma representación TF-IDF y el mismo conjunto de prueba.

### 11.1 Entrenamiento

In [29]:
from sklearn.naive_bayes import ComplementNB

modelo_cnb = ComplementNB()

modelo_cnb.fit(
    X_train_tfidf,
    y_train_exp
)

print("Modelo Complement Naive Bayes entrenado correctamente.")

Modelo Complement Naive Bayes entrenado correctamente.


### 11.2 Evaluación

In [30]:
y_pred_cnb = modelo_cnb.predict(
    X_test_tfidf
)

accuracy_cnb = accuracy_score(
    y_test,
    y_pred_cnb
)

f1_macro_cnb = f1_score(
    y_test,
    y_pred_cnb,
    average="macro"
)

print(
    "Accuracy Complement Naive Bayes:",
    round(accuracy_cnb, 4)
)

print(
    "F1 macro Complement Naive Bayes:",
    round(f1_macro_cnb, 4)
)

Accuracy Complement Naive Bayes: 0.8034
F1 macro Complement Naive Bayes: 0.5783


## 12. Cuarto modelo: SGDClassifier

Finalmente, se evalúa `SGDClassifier`, un clasificador lineal entrenado mediante descenso de gradiente estocástico.

Este algoritmo resulta especialmente interesante en problemas de clasificación de texto de gran escala, ya que permite trabajar eficientemente con conjuntos de datos de alta dimensionalidad y matrices dispersas como las generadas mediante TF-IDF.

Su inclusión permite analizar una alternativa computacionalmente eficiente y potencialmente escalable al conjunto completo de entrenamiento.

### 12.1 Entrenamiento

In [31]:
from sklearn.linear_model import SGDClassifier

modelo_sgd = SGDClassifier(
    loss="hinge",
    max_iter=1000,
    tol=1e-3,
    random_state=42
)

modelo_sgd.fit(
    X_train_tfidf,
    y_train_exp
)

print("Modelo SGDClassifier entrenado correctamente.")

Modelo SGDClassifier entrenado correctamente.


### 12.2 Evaluación

In [32]:
y_pred_sgd = modelo_sgd.predict(
    X_test_tfidf
)

accuracy_sgd = accuracy_score(
    y_test,
    y_pred_sgd
)

f1_macro_sgd = f1_score(
    y_test,
    y_pred_sgd,
    average="macro"
)

print(
    "Accuracy SGDClassifier:",
    round(accuracy_sgd, 4)
)

print(
    "F1 macro SGDClassifier:",
    round(f1_macro_sgd, 4)
)

Accuracy SGDClassifier: 0.8507
F1 macro SGDClassifier: 0.6546


## Comparación de todos los modelos

In [33]:
comparacion_modelos = pd.DataFrame({
    "Modelo": [
        "Baseline",
        "Regresión Logística",
        "Regresión Logística balanceada",
        "Linear SVM",
        "Complement Naive Bayes",
        "SGDClassifier"
    ],
    "Accuracy": [
        accuracy_baseline,
        accuracy_lr,
        accuracy_lr_balanced,
        accuracy_svm,
        accuracy_cnb,
        accuracy_sgd
    ],
    "F1 macro": [
        f1_macro_baseline,
        f1_macro_lr,
        f1_macro_lr_balanced,
        f1_macro_svm,
        f1_macro_cnb,
        f1_macro_sgd
    ]
})

comparacion_modelos[["Accuracy", "F1 macro"]] = (
    comparacion_modelos[["Accuracy", "F1 macro"]]
    .round(4)
)

comparacion_modelos.sort_values(
    "F1 macro",
    ascending=False
).reset_index(drop=True)

,Modelo,Accuracy,F1 macro
0,Linear SVM,0.8554,0.7118
1,Regresión Logística,0.8588,0.6966
2,Regresión Logística balanceada,0.8268,0.6943
3,SGDClassifier,0.8507,0.6546
4,Complement Naive Bayes,0.8034,0.5783
5,Baseline,0.5781,0.0666


### Selección del modelo

La comparación de los diferentes algoritmos muestra que todos los modelos supervisados evaluados superan al baseline, confirmando que la información contenida en las narrativas permite predecir la categoría de producto.

La Regresión Logística obtiene la mayor accuracy (0,8588), mientras que Linear SVM alcanza una accuracy muy similar (0,8554) y el mayor F1-score macro de todos los modelos evaluados (0,7118).

Por su parte, SGDClassifier presenta una accuracy de 0,8507, pero un F1-score macro inferior (0,6546). Complement Naive Bayes obtiene resultados más modestos, con una accuracy de 0,8034 y un F1-score macro de 0,5783.

Debido al desbalance existente entre las categorías, se prioriza el F1-score macro como criterio principal de selección, ya que permite evaluar el rendimiento otorgando el mismo peso a todas las clases. Bajo este criterio, **Linear SVM se selecciona como el mejor modelo de la fase experimental**.

A partir de esta selección, los siguientes experimentos se centrarán en analizar la capacidad de mejora de Linear SVM al incrementar el volumen de datos de entrenamiento.

## 13. Análisis del efecto del tamaño del conjunto de entrenamiento

Una vez seleccionado Linear SVM como el modelo con mejor F1-score macro, se analiza el efecto que tiene el volumen de datos de entrenamiento sobre su rendimiento.

Hasta este punto, los modelos se han entrenado utilizando una muestra estratificada de 100.000 observaciones del conjunto de entrenamiento. Sin embargo, el conjunto disponible contiene más de un millón de registros.

Para determinar si un mayor volumen de información permite mejorar la capacidad predictiva del modelo, se realizarán experimentos incrementales con diferentes tamaños de entrenamiento, manteniendo constante el conjunto de prueba.

Esta estrategia permite analizar tanto la evolución del rendimiento como el coste computacional asociado al incremento del volumen de datos.

### 13.1 Experimento con 250.000 registros

In [34]:
X_train_250k, _, y_train_250k, _ = train_test_split(
    X_train,
    y_train,
    train_size=250000,
    stratify=y_train,
    random_state=42
)

print("Tamaño de la muestra:", len(X_train_250k))

print("\nDistribución:")
print(
    y_train_250k
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Tamaño de la muestra: 250000

Distribución:
Product
Credit reporting or other personal consumer reports        57.81
Debt collection                                            12.51
Checking or savings account                                 7.99
Credit card                                                 7.49
Money transfer, virtual currency, or money service          4.77
Mortgage                                                    2.84
Vehicle loan or lease                                       2.03
Student loan                                                2.00
Payday loan, title loan, personal loan, or advance loan     1.38
Prepaid card                                                0.77
Debt or credit management                                   0.41
Name: proportion, dtype: float64


In [35]:
tfidf_250k = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000
)

X_train_250k_tfidf = tfidf_250k.fit_transform(
    X_train_250k
)

print(
    "Dimensiones TF-IDF entrenamiento:",
    X_train_250k_tfidf.shape
)

Dimensiones TF-IDF entrenamiento: (250000, 50000)


In [36]:
X_test_250k_tfidf = tfidf_250k.transform(
    X_test
)

print(
    "Dimensiones TF-IDF test:",
    X_test_250k_tfidf.shape
)

Dimensiones TF-IDF test: (259871, 50000)


### 13.2 Entrenamiento de Linear SVM con 250.000 observaciones

Una vez generada la representación TF-IDF correspondiente a la muestra de 250.000 observaciones, se entrena nuevamente Linear SVM manteniendo la misma configuración utilizada en el experimento inicial.

Además del rendimiento predictivo, se registra el tiempo de entrenamiento para analizar el coste computacional asociado al incremento del volumen de datos.

In [37]:
import time

inicio = time.time()

modelo_svm_250k = LinearSVC(
    random_state=42
)

modelo_svm_250k.fit(
    X_train_250k_tfidf,
    y_train_250k
)

tiempo_svm_250k = time.time() - inicio

print("Modelo Linear SVM entrenado correctamente.")
print(
    "Tiempo de entrenamiento:",
    round(tiempo_svm_250k, 2),
    "segundos"
)

Modelo Linear SVM entrenado correctamente.
Tiempo de entrenamiento: 164.94 segundos


### 13.3 Evaluación

In [38]:
y_pred_svm_250k = modelo_svm_250k.predict(
    X_test_250k_tfidf
)

accuracy_svm_250k = accuracy_score(
    y_test,
    y_pred_svm_250k
)

f1_macro_svm_250k = f1_score(
    y_test,
    y_pred_svm_250k,
    average="macro"
)

print(
    "Accuracy Linear SVM 250k:",
    round(accuracy_svm_250k, 4)
)

print(
    "F1 macro Linear SVM 250k:",
    round(f1_macro_svm_250k, 4)
)

Accuracy Linear SVM 250k: 0.859
F1 macro Linear SVM 250k: 0.7185


### Interpretación del experimento con 250.000 observaciones

El incremento del conjunto de entrenamiento de 100.000 a 250.000 observaciones produce una mejora en ambas métricas de evaluación.

La accuracy aumenta de 0,8554 a 0,8590, mientras que el F1-score macro pasa de 0,7118 a 0,7185. Este último resultado resulta especialmente relevante debido al desbalance existente entre las categorías.

El tiempo de entrenamiento de Linear SVM aumenta de aproximadamente 52 segundos a 164,94 segundos, como consecuencia del mayor volumen de información procesada.

Los resultados indican que el modelo continúa beneficiándose del incremento de datos de entrenamiento, aunque la mejora obtenida es moderada. Por este motivo, se realiza un nuevo experimento con 500.000 observaciones para analizar si el rendimiento continúa aumentando al ampliar el conjunto de entrenamiento.

### 13.4 Experimento con 500.000 registros

In [39]:
X_train_500k, _, y_train_500k, _ = train_test_split(
    X_train,
    y_train,
    train_size=500000,
    stratify=y_train,
    random_state=42
)

print("Tamaño de la muestra:", len(X_train_500k))

print("\nDistribución:")
print(
    y_train_500k
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Tamaño de la muestra: 500000

Distribución:
Product
Credit reporting or other personal consumer reports        57.81
Debt collection                                            12.51
Checking or savings account                                 7.99
Credit card                                                 7.49
Money transfer, virtual currency, or money service          4.77
Mortgage                                                    2.84
Vehicle loan or lease                                       2.03
Student loan                                                2.00
Payday loan, title loan, personal loan, or advance loan     1.38
Prepaid card                                                0.77
Debt or credit management                                   0.41
Name: proportion, dtype: float64


In [40]:
tfidf_500k = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000
)

X_train_500k_tfidf = tfidf_500k.fit_transform(
    X_train_500k
)

print(
    "Dimensiones TF-IDF entrenamiento:",
    X_train_500k_tfidf.shape
)

Dimensiones TF-IDF entrenamiento: (500000, 50000)


In [41]:
X_test_500k_tfidf = tfidf_500k.transform(
    X_test
)

print(
    "Dimensiones TF-IDF test:",
    X_test_500k_tfidf.shape
)

Dimensiones TF-IDF test: (259871, 50000)


### 13.5 Entrenamiento de Linear SVM con 500.000 observaciones

Se entrena Linear SVM utilizando la representación TF-IDF generada a partir de 500.000 observaciones. Se mantiene la misma configuración de los experimentos anteriores y se registra el tiempo de entrenamiento para evaluar el impacto computacional del incremento del conjunto de datos.

In [42]:
inicio = time.time()

modelo_svm_500k = LinearSVC(
    random_state=42
)

modelo_svm_500k.fit(
    X_train_500k_tfidf,
    y_train_500k
)

tiempo_svm_500k = time.time() - inicio

print("Modelo Linear SVM 500k entrenado correctamente.")
print(
    "Tiempo de entrenamiento:",
    round(tiempo_svm_500k, 2),
    "segundos"
)

Modelo Linear SVM 500k entrenado correctamente.
Tiempo de entrenamiento: 617.35 segundos


### 13.6 Evaluación

In [43]:
y_pred_svm_500k = modelo_svm_500k.predict(
    X_test_500k_tfidf
)

accuracy_svm_500k = accuracy_score(
    y_test,
    y_pred_svm_500k
)

f1_macro_svm_500k = f1_score(
    y_test,
    y_pred_svm_500k,
    average="macro"
)

print(
    "Accuracy Linear SVM 500k:",
    round(accuracy_svm_500k, 4)
)

print(
    "F1 macro Linear SVM 500k:",
    round(f1_macro_svm_500k, 4)
)

Accuracy Linear SVM 500k: 0.8643
F1 macro Linear SVM 500k: 0.7271


### Interpretación del experimento con 500.000 observaciones

El incremento del conjunto de entrenamiento hasta 500.000 observaciones vuelve a producir una mejora en el rendimiento de Linear SVM.

La accuracy aumenta de 0,8590 con 250.000 observaciones a 0,8643, mientras que el F1-score macro pasa de 0,7185 a 0,7271. En comparación con el experimento inicial de 100.000 observaciones, ambas métricas muestran una evolución positiva a medida que aumenta el volumen de datos utilizado durante el entrenamiento.

El coste computacional también aumenta de forma considerable. El entrenamiento del modelo con 500.000 observaciones requiere 617,35 segundos, aproximadamente 10,3 minutos.

Los resultados indican que el modelo continúa beneficiándose de un mayor volumen de información y que todavía no se observa una estabilización clara del rendimiento. Por este motivo, resulta pertinente evaluar finalmente el modelo utilizando todo el conjunto de entrenamiento disponible.

### 13.7 Liberación de memoria antes del entrenamiento final

Los experimentos anteriores han generado varias matrices TF-IDF de gran tamaño correspondientes a muestras de 100.000, 250.000 y 500.000 observaciones.

Dado que estas matrices ya no son necesarias para el entrenamiento final, se eliminan de memoria antes de procesar el conjunto completo. Esta decisión permite reducir el consumo de RAM y mejorar la estabilidad del entorno durante el experimento con más de un millón de observaciones.

In [44]:
import gc

variables_a_eliminar = [
    "X_train_tfidf",
    "X_test_tfidf",
    "X_train_250k_tfidf",
    "X_test_250k_tfidf",
    "X_train_500k_tfidf",
    "X_test_500k_tfidf"
]

for variable in variables_a_eliminar:
    if variable in globals():
        del globals()[variable]

gc.collect()

print("Memoria liberada.")

Memoria liberada.


### 13.8 TF-IDF con todo el entrenamiento

In [45]:
tfidf_full = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000
)

inicio = time.time()

X_train_full_tfidf = tfidf_full.fit_transform(
    X_train
)

tiempo_tfidf_full = time.time() - inicio

print(
    "Dimensiones TF-IDF entrenamiento completo:",
    X_train_full_tfidf.shape
)

print(
    "Tiempo TF-IDF entrenamiento:",
    round(tiempo_tfidf_full, 2),
    "segundos"
)

Dimensiones TF-IDF entrenamiento completo: (1039482, 50000)
Tiempo TF-IDF entrenamiento: 262.25 segundos


In [46]:
inicio = time.time()

X_test_full_tfidf = tfidf_full.transform(
    X_test
)

tiempo_tfidf_test_full = time.time() - inicio

print(
    "Dimensiones TF-IDF test:",
    X_test_full_tfidf.shape
)

print(
    "Tiempo transformación test:",
    round(tiempo_tfidf_test_full, 2),
    "segundos"
)

Dimensiones TF-IDF test: (259871, 50000)
Tiempo transformación test: 54.06 segundos


### 13.9 Entrenamiento de Linear SVM con el conjunto completo

Finalmente, se entrena Linear SVM utilizando las 1.039.482 observaciones disponibles en el conjunto de entrenamiento.

Este experimento permite evaluar el rendimiento alcanzable utilizando todo el corpus disponible y completar el análisis del efecto del tamaño del conjunto de entrenamiento. Se mantiene la misma configuración utilizada en los experimentos anteriores para garantizar la comparabilidad de los resultados.

In [47]:
inicio = time.time()

modelo_svm_full = LinearSVC(
    random_state=42
)

modelo_svm_full.fit(
    X_train_full_tfidf,
    y_train
)

tiempo_svm_full = time.time() - inicio

print("Modelo Linear SVM completo entrenado correctamente.")

print(
    "Tiempo de entrenamiento:",
    round(tiempo_svm_full, 2),
    "segundos"
)

print(
    "Tiempo de entrenamiento:",
    round(tiempo_svm_full / 60, 2),
    "minutos"
)

Modelo Linear SVM completo entrenado correctamente.
Tiempo de entrenamiento: 586.27 segundos
Tiempo de entrenamiento: 9.77 minutos


### 13.10 Evaluación final

In [48]:
y_pred_svm_full = modelo_svm_full.predict(
    X_test_full_tfidf
)

accuracy_svm_full = accuracy_score(
    y_test,
    y_pred_svm_full
)

f1_macro_svm_full = f1_score(
    y_test,
    y_pred_svm_full,
    average="macro"
)

print(
    "Accuracy Linear SVM completo:",
    round(accuracy_svm_full, 4)
)

print(
    "F1 macro Linear SVM completo:",
    round(f1_macro_svm_full, 4)
)

Accuracy Linear SVM completo: 0.8693
F1 macro Linear SVM completo: 0.7366


### Conclusión del análisis del tamaño de entrenamiento

Los experimentos realizados muestran una evolución positiva del rendimiento de Linear SVM a medida que aumenta el número de observaciones utilizadas durante el entrenamiento.

El F1-score macro aumenta progresivamente desde 0,7118 con 100.000 observaciones hasta 0,7366 utilizando las 1.039.482 observaciones disponibles. De forma paralela, la accuracy pasa de 0,8554 a 0,8693.

Estos resultados indican que el modelo se beneficia del incremento del volumen de información y que no se observa una estabilización completa del rendimiento dentro de los tamaños analizados. Por tanto, se selecciona el modelo Linear SVM entrenado con el conjunto completo como modelo final de esta fase.

El incremento del volumen de datos implica un mayor coste computacional, especialmente en las etapas de vectorización TF-IDF y entrenamiento. No obstante, este coste resulta asumible en el entorno utilizado y permite obtener el mejor rendimiento observado durante los experimentos.

In [49]:
evolucion_svm = pd.DataFrame({
    "Observaciones entrenamiento": [
        100000,
        250000,
        500000,
        len(X_train)
    ],
    "Accuracy": [
        accuracy_svm,
        accuracy_svm_250k,
        accuracy_svm_500k,
        accuracy_svm_full
    ],
    "F1 macro": [
        f1_macro_svm,
        f1_macro_svm_250k,
        f1_macro_svm_500k,
        f1_macro_svm_full
    ],
    "Tiempo entrenamiento (s)": [
        52.4,  # sustituye por el tiempo exacto de tu primer SVM si lo guardaste
        tiempo_svm_250k,
        tiempo_svm_500k,
        tiempo_svm_full
    ]
})

evolucion_svm[["Accuracy", "F1 macro"]] = (
    evolucion_svm[["Accuracy", "F1 macro"]]
    .round(4)
)

evolucion_svm["Tiempo entrenamiento (s)"] = (
    evolucion_svm["Tiempo entrenamiento (s)"]
    .round(2)
)

evolucion_svm

,Observaciones entrenamiento,Accuracy,F1 macro,Tiempo entrenamiento (s)
0,100000,0.8554,0.7118,52.40
1,250000,0.8590,0.7185,164.94
2,500000,0.8643,0.7271,617.35
3,1039482,0.8693,0.7366,586.27
